# Pose Review: Top 3 Compounds — TBXT Pocket D

Re-dock the 3 selected compounds from the pocket-D virtual screen with
high exhaustiveness and multiple poses for manual pose inspection.

In [ ]:
import pandas as pd
from tbxt_hackathon.docking import dock_single

## 1. Load selected compounds

In [ ]:
POCKET_PDB = "../data/structures/TGT_TBXT_D_pocket.pdb"

top3 = pd.read_csv("../data/docking_top3_pocket_D.csv")
assert top3.shape[0] == 3, f"Expected 3 compounds, got {top3.shape[0]}"
top3[["id", "smiles", "score", "LE"]]

## 2. Full docking (high exhaustiveness)

5 conformers, exhaustiveness=32, 5 poses per conformer.

In [ ]:
reports = []
for _, row in top3.iterrows():
    print(f"\nDocking {row['smiles'][:50]} ...")
    rpt = dock_single(
        row["smiles"], POCKET_PDB,
        n_conformers=5, exhaustiveness=32, n_poses=5,
    )
    reports.append((row.get("id", row.name), rpt))

## 3. Pose visualization

### Rank 1

In [ ]:
mol_id, rpt = reports[0]
print(f"{mol_id}: {rpt.score:.2f} kcal/mol  LE={rpt.ligand_efficiency:.3f}")
rpt.show_3d(show_labels=False)

### Rank 2

In [ ]:
mol_id, rpt = reports[1]
print(f"{mol_id}: {rpt.score:.2f} kcal/mol  LE={rpt.ligand_efficiency:.3f}")
rpt.show_3d(show_labels=False)

### Rank 3

In [ ]:
mol_id, rpt = reports[2]
print(f"{mol_id}: {rpt.score:.2f} kcal/mol  LE={rpt.ligand_efficiency:.3f}")
rpt.show_3d(show_labels=False)

## 4. Interaction comparison

In [ ]:
rows = []
for mol_id, rpt in reports:
    bp = rpt.best_pose
    rows.append({
        "id": mol_id,
        "smiles": rpt.smiles,
        "score": rpt.score,
        "LE": rpt.ligand_efficiency,
        "HA": rpt.n_heavy_atoms,
        "inter": bp.inter_energy,
        "intra": bp.intra_energy,
        "hbonds": sum(1 for x in bp.interactions if x.type == "hbond"),
        "hydrophobic": sum(1 for x in bp.interactions if x.type == "hydrophobic"),
        "pi_stack": sum(1 for x in bp.interactions if x.type == "pi_stacking"),
        "salt_bridge": sum(1 for x in bp.interactions if x.type == "salt_bridge"),
        "total_interactions": len(bp.interactions),
        "key_residues": ", ".join(sorted({x.protein_residue for x in bp.interactions})),
    })

comparison = pd.DataFrame(rows)
comparison.style.background_gradient(subset=["score"], cmap="RdYlGn_r")